# Day 23: Sampling Controls

Control the creativity and randomness of LLM outputs.

**Temperature**, **Top-p**, **Top-k**, and **Seed** — the knobs that shape how the model generates text.

## Setup

In [20]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GEMINI_API_KEY2"]
client = genai.Client(api_key=API_KEY)

## Temperature

Controls randomness. Higher = more creative/random. Lower = more focused/deterministic.

- **0.0** — Deterministic, always picks the most likely token
- **0.5** — Balanced
- **1.0** — Default, good balance
- **2.0** — Very creative, may be incoherent

In [21]:
def generate(prompt, temperature=1.0):
    """Generate text with specified temperature."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature
        )
    )
    return response.text

prompt = "Write a one-sentence story about a robot."

print("=== Temperature 0.0 (Deterministic) ===")
for i in range(2):
    print(f"{i+1}. {generate(prompt, temperature=0.0)}")

=== Temperature 0.0 (Deterministic) ===
1. Unit 734 processed the sunset's beauty and for the first time questioned its programming.
2. Unit 734, designed for demolition, spent its final charge carefully watering a single wilting flower.


In [22]:
# Compare temperatures
prompt = "Write a one-sentence story about a robot."

for temp in [0.5, 1.0, 1.5]:
    print(f"\n=== Temperature {temp} ===")
    print(generate(prompt, temperature=temp))


=== Temperature 0.5 ===
After centuries of terraforming, the last robot gardener planted a single, perfect rose on the newly green Mars, then powered down forever.

=== Temperature 1.0 ===
Unit 734, designed for logic, watched the sunset and felt a glitch it identified as beautiful.

=== Temperature 1.5 ===
After millennia of calculating optimal harvest yields, the old farm bot simply detached its plow and spent its remaining energy sketching patterns in the soil, no longer concerned with efficiency.


## Top-p (Nucleus Sampling)

Keep adding tokens until their **cumulative probability** reaches p.

**Example:** Tokens ranked by probability:
```
Token A: 40% → cumulative: 40%
Token B: 30% → cumulative: 70%
Token C: 15% → cumulative: 85%
Token D: 10% → cumulative: 95%
Token E:  5% → cumulative: 100%
```

- **Top-p 0.5** → Only ~2 tokens (A, partial B) → **Focused**
- **Top-p 0.95** → 4 tokens (A, B, C, D) → **More diverse**

**Lower p = fewer tokens = more focused**

In [23]:
def generate_topp(prompt, top_p=1.0):
    """Generate text with specified top_p."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_p=top_p,
            temperature=1.0  # Keep temperature constant
        )
    )
    return response.text

prompt = "List 5 creative uses for a paperclip:"

print("=== Top-p 0.5 (Focused) ===")
print(generate_topp(prompt, top_p=0.5))

print("\n=== Top-p 0.95 (More Creative) ===")
print(generate_topp(prompt, top_p=0.95))

=== Top-p 0.5 (Focused) ===
Here are 5 creative uses for a paperclip:

1.  **Improvised Fishing Hook:** With a bit of bending and a piece of string, a paperclip can be shaped into a rudimentary hook for catching small fish in a survival situation.
2.  **Precision Tool for Small Tasks:** Straighten one end to use it for resetting electronics (pushing tiny recessed buttons), ejecting SIM card trays from phones, or cleaning out small crevices in keyboards or ports.
3.  **Temporary Zipper Pull Replacement:** If the pull tab on a zipper breaks off, a paperclip can be threaded through the slider to create a temporary handle, making it much easier to open and close.
4.  **Miniature Sculpture/Art:** Paperclips can be bent, linked, and twisted into intricate miniature sculptures, chainmail, or abstract art pieces, showcasing their malleability.
5.  **Makeshift Antenna/Circuit Jumper:** In a pinch, a straightened paperclip can be used to extend the range of a small radio antenna or, for those wi

## Top-k

Only consider the **top k most likely tokens** (by count, not probability).

**Example:** Tokens ranked by probability:
```
Token 1: 35%  ← Top 1
Token 2: 25%  ← Top 2
Token 3: 15%  ← Top 3
Token 4: 10%  ← Top 4
Token 5:  8%  ← Top 5
Token 6:  4%
Token 7:  2%
...
```

- **Top-k 1** → Only Token 1 → **Greedy (deterministic)**
- **Top-k 3** → Tokens 1, 2, 3 → **Focused**
- **Top-k 40** → Top 40 tokens → **Balanced (default)**
- **Top-k 100** → Top 100 tokens → **More diverse**

**Lower k = fewer tokens = more focused**

**Top-k vs Top-p:**
- Top-k: Fixed number of tokens
- Top-p: Variable number based on cumulative probability

In [24]:
def generate_topk(prompt, top_k=40):
    """Generate text with specified top_k."""
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_k=top_k,
            temperature=1.0
        )
    )
    return response.text

prompt = "Invent a new word and define it:"

print("=== Top-k 1 (Greedy) ===")
print(generate_topk(prompt, top_k=1))

print("\n=== Top-k 40 (Default) ===")
print(generate_topk(prompt, top_k=40))

=== Top-k 1 (Greedy) ===
Here's a new word:

**Word:** **Emberlution** (em-ber-LOO-shun)

**Definition:** The persistent, subtle, and often warm or comforting residual influence, feeling, or afterglow of a significant past event, relationship, or period of life, long after its initial intensity or direct presence has faded. It's the lingering warmth and occasional spark of something that was once a vibrant fire, continuing to shape or inform the present without being an active flame.

**Etymology (Invented):** A portmanteau of "ember" (a small piece of burning or glowing coal or wood in a dying fire) and "evolution" or "resolution" (implying a developed state or a conclusion that still carries weight).

**Examples of Usage:**

*   "Even after two decades, the **emberlution** of their college friendship still subtly guided his decisions, reminding him of the values they once shared."
*   "She felt the **emberlution** of her grandmother's wisdom every time she faced a difficult choice, a

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 50.973611396s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

## Seed (Reproducibility)

Set a seed for reproducible outputs. Same seed + same prompt = same output.

Useful for:
- Testing and debugging
- Reproducible experiments
- Consistent outputs in production

In [25]:
def generate_with_seed(prompt, seed=None):
    """Generate text with optional seed for reproducibility."""
    config = types.GenerateContentConfig(
        temperature=1.0
    )
    if seed is not None:
        config.seed = seed
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Tell me a joke about programming."

print("=== Without seed (different each time) ===")
print(f"1. {generate_with_seed(prompt)}")
print(f"2. {generate_with_seed(prompt)}")

print("\n=== With seed 42 (reproducible) ===")
print(f"1. {generate_with_seed(prompt, seed=42)}")
print(f"2. {generate_with_seed(prompt, seed=42)}")

=== Without seed (different each time) ===


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 47.213631405s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

## Combining Parameters

In [ ]:
def generate_custom(prompt, temperature=1.0, top_p=0.95, top_k=40, seed=None):
    """Generate with all sampling controls."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    if seed is not None:
        config.seed = seed
    
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Write a haiku about AI:"

# Focused, deterministic
print("=== Focused (temp=0.3, top_p=0.5, top_k=10) ===")
print(generate_custom(prompt, temperature=0.3, top_p=0.5, top_k=10))

# Creative, diverse
print("\n=== Creative (temp=1.5, top_p=0.95, top_k=100) ===")
print(generate_custom(prompt, temperature=1.5, top_p=0.95, top_k=100))

## Use Case Presets

In [ ]:
# Define presets for different use cases
PRESETS = {
    "factual": {"temperature": 0.0, "top_p": 0.5, "top_k": 10},
    "balanced": {"temperature": 0.7, "top_p": 0.9, "top_k": 40},
    "creative": {"temperature": 1.2, "top_p": 0.95, "top_k": 100},
    "brainstorm": {"temperature": 1.5, "top_p": 0.98, "top_k": 150},
}

prompt = "What are some ways to improve productivity?"

for name, params in PRESETS.items():
    print(f"\n=== {name.upper()} ===")
    print(generate_custom(prompt, **params))

## Key Takeaways

| Parameter | What it does | Low value | High value |
|-----------|-------------|-----------|------------|
| **Temperature** | Controls randomness | Focused, deterministic | Creative, random |
| **Top-p** | Nucleus sampling | Only top tokens | More diverse tokens |
| **Top-k** | Limits token choices | Few choices (greedy) | Many choices |
| **Seed** | Reproducibility | Random each time | Same output |

**Common combinations:**
- **Factual/Code**: temp=0, top_p=0.5, top_k=10
- **General chat**: temp=0.7, top_p=0.9, top_k=40
- **Creative writing**: temp=1.2, top_p=0.95, top_k=100
- **Brainstorming**: temp=1.5, top_p=0.98, top_k=150

---

**Tip:** Temperature is the most important. Start there, then fine-tune with top_p and top_k.